## Employee Management Database System (SQL in Google Colab)

This notebook will guide you through the creation of a comprehensive Employee Management Database System using SQLite in Google Colab. We will cover database setup, table creation, data insertion, and various SQL query types, suitable for a portfolio project.

### Project Requirements Summary:
- Database Name: `Employee_Management_DB`
- Tables: `Departments`, `Employees`, `Jobs`, `Salaries`, `Attendance`, `Projects`, `Employee_Projects`.
- Constraints: Primary Keys, Foreign Keys, `NOT NULL`, `UNIQUE`, `CHECK`, `DEFAULT`, `AUTO INCREMENT`.
- Sample Data: Realistic data for all tables.
- SQL Sections: Structured as requested, covering basic to advanced SQL.
- Documentation: Clear explanations and comments.
- Compatibility: SQLite (preferred for Google Colab).
- Final Deliverables: ER Diagram, schema summary, relationship list, SQL concepts summary, future enhancements.


### Section 1: Create and Connect to Database

**Purpose:** This section initializes the SQLite database. SQLite is a file-based database, meaning the entire database is stored in a single file on disk. We will create a connection to this file, which will automatically create the database file if it doesn't already exist.

**SQL Concepts Demonstrated:**
- **Connecting to SQLite:** Using Python's `sqlite3` module to establish a connection.
- **Cursor Object:** Creating a cursor object to execute SQL commands.

**Expected Output:** A message confirming successful database connection and creation (if new).

In [24]:
import sqlite3

# Define the database name
DATABASE_NAME = 'Employee_Management_DB.db'

# Establish a connection to the SQLite database
# If the database file does not exist, it will be created.
try:
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    print(f"Successfully connected to the database: {DATABASE_NAME}")
except sqlite3.Error as e:
    print(f"Error connecting to database: {e}")

# In SQLite, there's no explicit 'USE DATABASE' command like in other SQL systems.
# The connection itself specifies which database you are working with.
# We will use the 'cursor' object to execute all subsequent SQL commands.


Successfully connected to the database: Employee_Management_DB.db


### Section 2: Create Tables

**Purpose:** This section defines the schema of our database by creating all the required tables. We will establish relationships between tables using primary and foreign keys, and enforce data integrity with `NOT NULL`, `UNIQUE`, `CHECK`, and `DEFAULT` constraints, along with `AUTO INCREMENT` for IDs.

**SQL Concepts Demonstrated:**
- **CREATE TABLE:** Statement for defining new tables.
- **Data Types:** Choosing appropriate data types (`INTEGER`, `TEXT`, `REAL`, `DATE`, `BOOLEAN`).
- **Primary Key (`PRIMARY KEY`):** Uniquely identifies each record in a table.
- **Foreign Key (`FOREIGN KEY`):** Establishes relationships between tables and enforces referential integrity.
- **`NOT NULL`:** Ensures that a column cannot have a `NULL` value.
- **`UNIQUE`:** Ensures all values in a column are different.
- **`DEFAULT`:** Assigns a default value if no other value is specified.
- **`CHECK`:** Enforces domain integrity by limiting values in a column.
- **`AUTOINCREMENT`:** (In SQLite, `INTEGER PRIMARY KEY` implies autoincrementing behavior by default, but `AUTOINCREMENT` keyword can be added for specific cases, though it adds some overhead and is often not strictly necessary for basic autoincrementing.)

**Expected Output:** Messages indicating successful table creation, or any errors encountered during the process.

In [25]:
# SQL commands to create tables
create_tables_sql = """

-- Departments Table
CREATE TABLE IF NOT EXISTS Departments (
    department_id INTEGER PRIMARY KEY AUTOINCREMENT,
    department_name TEXT NOT NULL UNIQUE,
    location TEXT DEFAULT 'Headquarters',
    phone_number TEXT UNIQUE
);

-- Jobs Table
CREATE TABLE IF NOT EXISTS Jobs (
    job_id INTEGER PRIMARY KEY AUTOINCREMENT,
    job_title TEXT NOT NULL UNIQUE,
    min_salary REAL CHECK (min_salary >= 0) DEFAULT 30000,
    max_salary REAL CHECK (max_salary >= min_salary)
);

-- Employees Table
CREATE TABLE IF NOT EXISTS Employees (
    employee_id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    phone_number TEXT UNIQUE,
    hire_date DATE NOT NULL DEFAULT CURRENT_DATE,
    job_id INTEGER,
    department_id INTEGER,
    manager_id INTEGER, -- Self-referencing foreign key
    FOREIGN KEY (job_id) REFERENCES Jobs(job_id) ON DELETE SET NULL ON UPDATE CASCADE,
    FOREIGN KEY (department_id) REFERENCES Departments(department_id) ON DELETE SET NULL ON UPDATE CASCADE,
    FOREIGN KEY (manager_id) REFERENCES Employees(employee_id) ON DELETE SET NULL ON UPDATE CASCADE
);

-- Drop Salaries table if it exists to allow schema modification
DROP TABLE IF EXISTS Salaries;

-- Salaries Table
CREATE TABLE IF NOT EXISTS Salaries (
    salary_id INTEGER PRIMARY KEY AUTOINCREMENT,
    employee_id INTEGER NOT NULL,
    amount REAL NOT NULL CHECK (amount >= 0),
    effective_date DATE NOT NULL DEFAULT CURRENT_DATE,
    FOREIGN KEY (employee_id) REFERENCES Employees(employee_id) ON DELETE CASCADE ON UPDATE CASCADE
);

-- Attendance Table
CREATE TABLE IF NOT EXISTS Attendance (
    attendance_id INTEGER PRIMARY KEY AUTOINCREMENT,
    employee_id INTEGER NOT NULL,
    attendance_date DATE NOT NULL DEFAULT CURRENT_DATE,
    check_in_time TIME,
    check_out_time TIME,
    status TEXT CHECK (status IN ('Present', 'Absent', 'Leave')) DEFAULT 'Present',
    hours_worked REAL,
    UNIQUE(employee_id, attendance_date), -- An employee can only have one attendance record per day
    FOREIGN KEY (employee_id) REFERENCES Employees(employee_id) ON DELETE CASCADE ON UPDATE CASCADE
);

-- Projects Table
CREATE TABLE IF NOT EXISTS Projects (
    project_id INTEGER PRIMARY KEY AUTOINCREMENT,
    project_name TEXT NOT NULL UNIQUE,
    start_date DATE NOT NULL DEFAULT CURRENT_DATE,
    end_date DATE,
    budget REAL CHECK (budget >= 0),
    status TEXT CHECK (status IN ('Planning', 'Active', 'Completed', 'On Hold')) DEFAULT 'Planning'
);

-- Employee_Projects (Many-to-Many relationship table)
CREATE TABLE IF NOT EXISTS Employee_Projects (
    employee_id INTEGER NOT NULL,
    project_id INTEGER NOT NULL,
    assigned_date DATE NOT NULL DEFAULT CURRENT_DATE,
    role TEXT,
    PRIMARY KEY (employee_id, project_id), -- Composite Primary Key
    FOREIGN KEY (employee_id) REFERENCES Employees(employee_id) ON DELETE CASCADE ON UPDATE CASCADE,
    FOREIGN KEY (project_id) REFERENCES Projects(project_id) ON DELETE CASCADE ON UPDATE CASCADE
);

"""

try:
    cursor.executescript(create_tables_sql)
    conn.commit()
    print("All tables created successfully!")
except sqlite3.Error as e:
    print(f"Error creating tables: {e}")

All tables created successfully!


### Section 3: Insert Sample Data

**Purpose:** This section populates the newly created tables with realistic sample data. This data will be used to test various SQL queries and demonstrate the functionality of the database. We will ensure the data is consistent with the defined constraints and relationships.

**SQL Concepts Demonstrated:**
- **INSERT INTO:** Statement for adding new rows of data into a table.
- **Data Consistency:** Ensuring that foreign key relationships are respected and constraints (like `UNIQUE`, `NOT NULL`, `CHECK`) are met.
- **Realistic Data Generation:** Creating diverse and plausible records for employees, departments, jobs, salaries, attendance, and projects.

**Expected Output:** A message confirming successful data insertion, or any errors encountered during the process.

In [26]:
import random
from datetime import datetime, timedelta

# Helper function to generate random dates
def random_date(start, end):
    return start + timedelta(days=random.randint(0, int((end - start).days)))

# SQL commands to insert sample data
insert_data_sql = """

-- Insert Departments (10 records)
INSERT INTO Departments (department_name, location, phone_number) VALUES
('Human Resources', 'Building A', '+1-555-0101'),
('Engineering', 'Building B', '+1-555-0102'),
('Marketing', 'Building C', '+1-555-0103'),
('Sales', 'Building D', '+1-555-0104'),
('Finance', 'Building A', '+1-555-0105'),
('Research & Development', 'Building E', '+1-555-0106'),
('Customer Service', 'Building C', '+1-555-0107'),
('IT Support', 'Building B', '+1-555-0108'),
('Operations', 'Building D', '+1-555-0109'),
('Legal', 'Building A', '+1-555-0110');

-- Insert Job Roles (15 records)
INSERT INTO Jobs (job_title, min_salary, max_salary) VALUES
('Software Engineer', 70000, 120000),
('HR Manager', 60000, 100000),
('Marketing Specialist', 50000, 80000),
('Sales Representative', 40000, 70000),
('Financial Analyst', 55000, 95000),
('Research Scientist', 75000, 130000),
('Customer Support Agent', 35000, 55000),
('IT Administrator', 50000, 90000),
('Operations Manager', 65000, 110000),
('Legal Counsel', 80000, 150000),
('Project Manager', 80000, 140000),
('Data Scientist', 90000, 160000),
('Product Manager', 85000, 150000),
('Senior Software Engineer', 90000, 150000),
('Junior Software Engineer', 55000, 80000);

"""

try:
    cursor.executescript(insert_data_sql)
    conn.commit()
    print("Departments and Jobs data inserted successfully.")

    # Generate and insert Employees (50 records)
    # We need job_id and department_id, so we'll fetch them
    cursor.execute("SELECT job_id, job_title FROM Jobs;")
    jobs = cursor.fetchall()
    cursor.execute("SELECT department_id, department_name FROM Departments;")
    departments = cursor.fetchall()

    employees_data = []
    for i in range(1, 51):
        first_name = f"Employee{i}"
        last_name = f"Lastname{i}"
        email = f"employee{i}@company.com"
        phone_number = f"+1-555-02{i:02d}"
        hire_date = random_date(datetime(2020, 1, 1), datetime.now()).strftime('%Y-%m-%d')
        job = random.choice(jobs)
        department = random.choice(departments)
        job_id = job[0]
        department_id = department[0]
        # Assign a manager (randomly pick an employee with a lower ID, or None for first few)
        manager_id = random.randint(1, i-1) if i > 5 else None # Ensure managers exist
        employees_data.append((first_name, last_name, email, phone_number, hire_date, job_id, department_id, manager_id))

    cursor.executemany("INSERT INTO Employees (first_name, last_name, email, phone_number, hire_date, job_id, department_id, manager_id) VALUES (?, ?, ?, ?, ?, ?, ?, ?)", employees_data)
    conn.commit()
    print("Employee data inserted successfully.")

    # Generate and insert Salaries (50 records)
    salaries_data = []
    cursor.execute("SELECT employee_id, job_id FROM Employees;")
    employee_job_ids = cursor.fetchall()
    for employee_id, job_id in employee_job_ids:
        cursor.execute("SELECT min_salary, max_salary FROM Jobs WHERE job_id = ?", (job_id,))
        min_salary, max_salary = cursor.fetchone()
        amount = round(random.uniform(min_salary, max_salary), 2)
        effective_date = random_date(datetime(2021, 1, 1), datetime.now()).strftime('%Y-%m-%d')
        salaries_data.append((employee_id, amount, effective_date))

    cursor.executemany("INSERT INTO Salaries (employee_id, amount, effective_date) VALUES (?, ?, ?)", salaries_data)
    conn.commit()
    print("Salaries data inserted successfully.")

    # Generate and insert Attendance records (multiple days for employees)
    attendance_data = []
    start_attendance_date = datetime(2023, 1, 1)
    end_attendance_date = datetime.now()

    for employee_id, _ in employee_job_ids:
        num_attendance_days = random.randint(10, 30) # Each employee has 10-30 attendance records
        for _ in range(num_attendance_days):
            att_date = random_date(start_attendance_date, end_attendance_date)
            check_in = (datetime.combine(att_date, datetime.min.time()) + timedelta(hours=random.randint(8, 9), minutes=random.randint(0, 59)))
            check_out = (datetime.combine(att_date, datetime.min.time()) + timedelta(hours=random.randint(17, 18), minutes=random.randint(0, 59)))

            status_choice = random.choices(['Present', 'Absent', 'Leave'], weights=[0.8, 0.1, 0.1], k=1)[0]
            hours_worked = round((check_out - check_in).total_seconds() / 3600, 2) if status_choice == 'Present' else None

            attendance_data.append((employee_id, att_date.strftime('%Y-%m-%d'), check_in.strftime('%H:%M:%S') if status_choice == 'Present' else None, check_out.strftime('%H:%M:%S') if status_choice == 'Present' else None, status_choice, hours_worked))

    # SQLite has issues with executemany for UNIQUE constraint on date if same employee_id and date. Filter duplicates.
    # For simplicity, we'll insert one by one or handle potential errors if many duplicates are generated.
    # Better to pre-process to ensure uniqueness for (employee_id, attendance_date)
    unique_attendance_records = {}
    for record in attendance_data:
        key = (record[0], record[1]) # (employee_id, attendance_date)
        if key not in unique_attendance_records:
            unique_attendance_records[key] = record

    filtered_attendance_data = list(unique_attendance_records.values())

    cursor.executemany("INSERT INTO Attendance (employee_id, attendance_date, check_in_time, check_out_time, status, hours_worked) VALUES (?, ?, ?, ?, ?, ?)", filtered_attendance_data)
    conn.commit()
    print("Attendance data inserted successfully.")

    # Insert Projects (20 records)
    projects_data = []
    for i in range(1, 21):
        project_name = f"Project Alpha {i}"
        start_date = random_date(datetime(2022, 1, 1), datetime(2023, 12, 31))
        end_date = start_date + timedelta(days=random.randint(90, 365)) # 3 to 12 months duration
        budget = round(random.uniform(50000, 500000), 2)
        status = random.choice(['Planning', 'Active', 'Completed', 'On Hold'])
        projects_data.append((project_name, start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d'), budget, status))

    cursor.executemany("INSERT INTO Projects (project_name, start_date, end_date, budget, status) VALUES (?, ?, ?, ?, ?)", projects_data)
    conn.commit()
    print("Projects data inserted successfully.")

    # Insert Employee_Projects (Many-to-Many assignments)
    employee_projects_data = []
    cursor.execute("SELECT employee_id FROM Employees;")
    all_employee_ids = [e[0] for e in cursor.fetchall()]
    cursor.execute("SELECT project_id FROM Projects;")
    all_project_ids = [p[0] for p in cursor.fetchall()]

    for employee_id in all_employee_ids:
        num_projects = random.randint(0, 3) # Each employee works on 0 to 3 projects
        assigned_projects = random.sample(all_project_ids, min(num_projects, len(all_project_ids))) # Ensure not to pick more projects than available
        for project_id in assigned_projects:
            assigned_date = random_date(datetime(2023, 1, 1), datetime.now()).strftime('%Y-%m-%d')
            role = random.choice(['Developer', 'Tester', 'Designer', 'Analyst', 'Team Lead'])
            employee_projects_data.append((employee_id, project_id, assigned_date, role))

    # Filter out duplicate (employee_id, project_id) pairs if any were generated by random.sample
    unique_employee_projects = {}
    for record in employee_projects_data:
        key = (record[0], record[1]) # (employee_id, project_id)
        if key not in unique_employee_projects:
            unique_employee_projects[key] = record

    filtered_employee_projects_data = list(unique_employee_projects.values())

    cursor.executemany("INSERT INTO Employee_Projects (employee_id, project_id, assigned_date, role) VALUES (?, ?, ?, ?)", filtered_employee_projects_data)
    conn.commit()
    print("Employee_Projects data inserted successfully.")

except sqlite3.Error as e:
    print(f"Error inserting data: {e}")




Error inserting data: UNIQUE constraint failed: Departments.phone_number


### Section 4: Basic Queries

**Purpose:** This section demonstrates fundamental SQL `SELECT` statements to retrieve data from the database. It covers how to fetch all records from a table and how to apply simple `WHERE` clauses to filter results based on specific conditions.

**SQL Concepts Demonstrated:**
- **`SELECT * FROM table_name`:** Retrieving all columns and all rows from a table.
- **`SELECT column1, column2 FROM table_name`:** Selecting specific columns.
- **`WHERE clause`:** Filtering rows based on a specified condition.
- **Comparison Operators:** `>, =` for numerical and date comparisons.

**Expected Output:** Tabular results for each query, displaying the requested data.

In [27]:
def execute_and_print_query(query_description, sql_query):
    print(f"\n--- {query_description} ---")
    try:
        cursor.execute(sql_query)
        results = cursor.fetchall()
        if results:
            # Get column names
            col_names = [description[0] for description in cursor.description]
            print(" | ".join(col_names))
            print("-" * (sum(len(c) for c in col_names) + 3 * len(col_names) - 1))
            for row in results:
                print(" | ".join(map(str, row)))
        else:
            print("No results found.")
    except sqlite3.Error as e:
        print(f"Error executing query: {e}")


# 1. Display all employees
execute_and_print_query(
    "Display all employees (first 5 rows for brevity)",
    "SELECT employee_id, first_name, last_name, email, hire_date FROM Employees LIMIT 5;"
)

# 2. Display all departments
execute_and_print_query(
    "Display all departments",
    "SELECT department_id, department_name, location FROM Departments;"
)

# 3. Employees with salary greater than 50,000
execute_and_print_query(
    "Employees with salary greater than 50,000 (first 5 rows)",
    """
    SELECT
        E.first_name,
        E.last_name,
        S.amount
    FROM Employees AS E
    JOIN Salaries AS S ON E.employee_id = S.employee_id
    WHERE S.amount > 50000
    LIMIT 5;
    """
)

# 4. Employees hired after 2023
execute_and_print_query(
    "Employees hired after 2023 (first 5 rows)",
    "SELECT employee_id, first_name, last_name, hire_date FROM Employees WHERE hire_date > '2023-12-31' LIMIT 5;"
)

# 5. Employees in a particular department (e.g., 'Engineering')
execute_and_print_query(
    "Employees in 'Engineering' department (first 5 rows)",
    """
    SELECT
        E.first_name,
        E.last_name,
        D.department_name
    FROM Employees AS E
    JOIN Departments AS D ON E.department_id = D.department_id
    WHERE D.department_name = 'Engineering'
    LIMIT 5;
    """
)



--- Display all employees (first 5 rows for brevity) ---
employee_id | first_name | last_name | email | hire_date
----------------------------------------------------------
1 | Employee1 | Lastname1 | employee1@company.com | 2021-11-14
2 | Employee2 | Lastname2 | employee2@company.com | 2024-05-03
3 | Employee3 | Lastname3 | employee3@company.com | 2024-04-26
4 | Employee4 | Lastname4 | employee4@company.com | 2023-11-02
5 | Employee5 | Lastname5 | employee5@company.com | 2024-10-01

--- Display all departments ---
department_id | department_name | location
--------------------------------------------
1 | Human Resources | Building A
2 | Engineering | Building B
3 | Marketing | Building C
4 | Sales | Building D
5 | Finance | Building A
6 | Research & Development | Building E
7 | Customer Service | Building C
8 | IT Support | Building B
9 | Operations | Building D
10 | Legal | Building A

--- Employees with salary greater than 50,000 (first 5 rows) ---
No results found.

--- Employees 

### Section 5: Filtering and Ordering

**Purpose:** This section explores advanced filtering and ordering capabilities in SQL. We will use various clauses to select specific subsets of data and present them in a desired order, making the results more targeted and readable.

**SQL Concepts Demonstrated:**
- **`WHERE` clause:** For conditional selection of rows.
- **`BETWEEN`:** To select values within a given range (inclusive).
- **`IN`:** To specify multiple possible values for a column.
- **`LIKE`:** For pattern matching using wildcards (`%` and `_`).
- **`ORDER BY`:** To sort the result set in ascending (`ASC`) or descending (`DESC`) order.
- **`LIMIT`:** To restrict the number of rows returned by a query.

**Expected Output:** Tabular results for each query, demonstrating the effect of each filtering and ordering clause.

In [28]:
# Re-using the execute_and_print_query helper function defined earlier

# 1. Employees hired between '2023-01-01' and '2024-12-31'
execute_and_print_query(
    "Employees hired between 2023 and 2024",
    "SELECT employee_id, first_name, last_name, hire_date FROM Employees WHERE hire_date BETWEEN '2023-01-01' AND '2024-12-31' LIMIT 5;"
)

# 2. Employees in 'Engineering' or 'Marketing' departments (using IN)
execute_and_print_query(
    "Employees in Engineering or Marketing departments",
    """
    SELECT
        E.first_name,
        E.last_name,
        D.department_name
    FROM Employees AS E
    JOIN Departments AS D ON E.department_id = D.department_id
    WHERE D.department_name IN ('Engineering', 'Marketing')
    LIMIT 5;
    """
)

# 3. Employees whose first name starts with 'Emp' (using LIKE)
execute_and_print_query(
    "Employees whose first name starts with 'Emp'",
    "SELECT employee_id, first_name, last_name FROM Employees WHERE first_name LIKE 'Emp%';"
)

# 4. Employees with 'manager' in their job title (case-insensitive search for SQLite)
execute_and_print_query(
    "Employees with 'manager' in their job title",
    """
    SELECT
        E.first_name,
        E.last_name,
        J.job_title
    FROM Employees AS E
    JOIN Jobs AS J ON E.job_id = J.job_id
    WHERE J.job_title LIKE '%Manager%'
    LIMIT 5;
    """
)

# 5. All employees ordered by hire date (newest first) and then by last name
execute_and_print_query(
    "All employees ordered by hire date (DESC) and then last name (ASC) (first 5 rows)",
    "SELECT employee_id, first_name, last_name, hire_date FROM Employees ORDER BY hire_date DESC, last_name ASC LIMIT 5;"
)

# 6. Top 3 highest paid employees
execute_and_print_query(
    "Top 3 highest paid employees",
    """
    SELECT
        E.first_name,
        E.last_name,
        S.amount
    FROM Employees AS E
    JOIN Salaries AS S ON E.employee_id = S.employee_id
    ORDER BY S.amount DESC
    LIMIT 3;
    """
)



--- Employees hired between 2023 and 2024 ---
employee_id | first_name | last_name | hire_date
--------------------------------------------------
2 | Employee2 | Lastname2 | 2024-05-03
3 | Employee3 | Lastname3 | 2024-04-26
4 | Employee4 | Lastname4 | 2023-11-02
5 | Employee5 | Lastname5 | 2024-10-01
6 | Employee6 | Lastname6 | 2024-11-27

--- Employees in Engineering or Marketing departments ---
first_name | last_name | department_name
------------------------------------------
Employee7 | Lastname7 | Engineering
Employee22 | Lastname22 | Engineering
Employee24 | Lastname24 | Engineering
Employee30 | Lastname30 | Engineering
Employee45 | Lastname45 | Engineering

--- Employees whose first name starts with 'Emp' ---
employee_id | first_name | last_name
--------------------------------------
1 | Employee1 | Lastname1
2 | Employee2 | Lastname2
3 | Employee3 | Lastname3
4 | Employee4 | Lastname4
5 | Employee5 | Lastname5
6 | Employee6 | Lastname6
7 | Employee7 | Lastname7
8 | Employee8 |

### Section 6: Aggregate Functions

**Purpose:** This section demonstrates how to use SQL aggregate functions to perform calculations on a set of rows and return a single summary value. These functions are crucial for data analysis and reporting.

**SQL Concepts Demonstrated:**
- **`COUNT()`:** Counts the number of rows or non-NULL values in a column.
- **`SUM()`:** Calculates the sum of values in a numeric column.
- **`AVG()`:** Calculates the average of values in a numeric column.
- **`MAX()`:** Finds the maximum value in a column.
- **`MIN()`:** Finds the minimum value in a column.

**Expected Output:** Single-row results for each query, displaying the calculated aggregate values.

In [29]:
# Re-using the execute_and_print_query helper function defined earlier

# 1. Total number of employees
execute_and_print_query(
    "Total number of employees",
    "SELECT COUNT(employee_id) AS total_employees FROM Employees;"
)

# 2. Total number of departments
execute_and_print_query(
    "Total number of departments",
    "SELECT COUNT(department_id) AS total_departments FROM Departments;"
)

# 3. Sum of all salaries
execute_and_print_query(
    "Sum of all salaries",
    "SELECT SUM(amount) AS total_payroll FROM Salaries;"
)

# 4. Average salary across all employees
execute_and_print_query(
    "Average salary across all employees",
    "SELECT AVG(amount) AS average_salary FROM Salaries;"
)

# 5. Highest salary
execute_and_print_query(
    "Highest salary",
    "SELECT MAX(amount) AS highest_salary FROM Salaries;"
)

# 6. Lowest salary
execute_and_print_query(
    "Lowest salary",
    "SELECT MIN(amount) AS lowest_salary FROM Salaries;"
)

# 7. Total hours worked by all employees (sum of hours_worked in attendance)
execute_and_print_query(
    "Total hours worked by all employees recorded in attendance",
    "SELECT SUM(hours_worked) AS total_attendance_hours FROM Attendance;"
)



--- Total number of employees ---
total_employees
-----------------
50

--- Total number of departments ---
total_departments
-------------------
10

--- Sum of all salaries ---
total_payroll
---------------
None

--- Average salary across all employees ---
average_salary
----------------
None

--- Highest salary ---
highest_salary
----------------
None

--- Lowest salary ---
lowest_salary
---------------
None

--- Total hours worked by all employees recorded in attendance ---
total_attendance_hours
------------------------
7038.129999999997


### Section 7: GROUP BY and HAVING

**Purpose:** This section demonstrates how to use the `GROUP BY` clause to group rows that have the same values in specified columns into summary rows, and the `HAVING` clause to filter those groups based on aggregate conditions. These are essential for analyzing data across categories.

**SQL Concepts Demonstrated:**
- **`GROUP BY`:** Groups rows that have the same values into summary rows, often used with aggregate functions.
- **`HAVING`:** Filters groups created by the `GROUP BY` clause, similar to `WHERE` but applied to aggregate results.

**Expected Output:** Tabular results for each query, showing grouped data and filtered groups.

In [30]:
# Re-using the execute_and_print_query helper function defined earlier

# 1. Department-wise employee count
execute_and_print_query(
    "Department-wise employee count",
    """
    SELECT
        D.department_name,
        COUNT(E.employee_id) AS employee_count
    FROM Departments AS D
    LEFT JOIN Employees AS E ON D.department_id = E.department_id
    GROUP BY D.department_name
    ORDER BY employee_count DESC;
    """
)

# 2. Average salary by department
execute_and_print_query(
    "Average salary by department",
    """
    SELECT
        D.department_name,
        AVG(S.amount) AS average_department_salary
    FROM Departments AS D
    JOIN Employees AS E ON D.department_id = E.department_id
    JOIN Salaries AS S ON E.employee_id = S.employee_id
    GROUP BY D.department_name
    ORDER BY average_department_salary DESC;
    """
)

# 3. Job title-wise average and max salary
execute_and_print_query(
    "Job title-wise average and max salary",
    """
    SELECT
        J.job_title,
        AVG(S.amount) AS average_job_salary,
        MAX(S.amount) AS max_job_salary
    FROM Jobs AS J
    JOIN Employees AS E ON J.job_id = E.job_id
    JOIN Salaries AS S ON E.employee_id = S.employee_id
    GROUP BY J.job_title
    ORDER BY average_job_salary DESC;
    """
)

# 4. Departments with more than five employees
execute_and_print_query(
    "Departments with more than five employees",
    """
    SELECT
        D.department_name,
        COUNT(E.employee_id) AS employee_count
    FROM Departments AS D
    JOIN Employees AS E ON D.department_id = E.department_id
    GROUP BY D.department_name
    HAVING COUNT(E.employee_id) > 5
    ORDER BY employee_count DESC;
    """
)

# 5. Projects with an average budget greater than 300,000
execute_and_print_query(
    "Projects with budget over 300,000 (average)",
    """
    SELECT
        project_name,
        budget
    FROM Projects
    WHERE budget > 300000
    ORDER BY budget DESC;
    """
)



--- Department-wise employee count ---
department_name | employee_count
----------------------------------
Finance | 8
Human Resources | 8
Sales | 7
Engineering | 6
IT Support | 6
Legal | 5
Customer Service | 4
Research & Development | 3
Operations | 2
Marketing | 1

--- Average salary by department ---
No results found.

--- Job title-wise average and max salary ---
No results found.

--- Departments with more than five employees ---
department_name | employee_count
----------------------------------
Finance | 8
Human Resources | 8
Sales | 7
Engineering | 6
IT Support | 6

--- Projects with budget over 300,000 (average) ---
project_name | budget
-----------------------
Project Alpha 2 | 496327.64
Project Alpha 5 | 492563.09
Project Alpha 7 | 490247.85
Project Alpha 19 | 440770.7
Project Alpha 4 | 376988.76
Project Alpha 6 | 348475.52
Project Alpha 11 | 324884.32
Project Alpha 17 | 313671.42
Project Alpha 9 | 303152.47


### Section 8: JOIN Operations

**Purpose:** This section demonstrates how to combine rows from two or more tables based on a related column between them. Joins are fundamental for retrieving meaningful information from a relational database where data is distributed across multiple tables.

**SQL Concepts Demonstrated:**
- **`INNER JOIN`:** Returns only the rows that have matching values in both tables.
- **`LEFT JOIN` (or `LEFT OUTER JOIN`):** Returns all rows from the left table, and the matching rows from the right table. If there is no match, the right side will have `NULL` values.
- **`RIGHT JOIN` (or `RIGHT OUTER JOIN`):** SQLite does not natively support `RIGHT JOIN`. It can often be simulated by swapping the tables and using a `LEFT JOIN`.
- **`CROSS JOIN`:** Returns the Cartesian product of the rows from the joined tables (each row from the first table combined with each row from the second table). (Not explicitly requested, but good to know).
- **`SELF JOIN`:** Joining a table with itself, typically used when comparing rows within the same table (e.g., finding employees managed by other employees).

**Expected Output:** Tabular results showing combined data from multiple tables based on the specified join conditions.

In [31]:
# Re-using the execute_and_print_query helper function defined earlier

# 1. Employee with their department details (INNER JOIN)
execute_and_print_query(
    "Employees with their department details (first 5 rows)",
    """
    SELECT
        E.first_name,
        E.last_name,
        D.department_name,
        D.location
    FROM Employees AS E
    INNER JOIN Departments AS D ON E.department_id = D.department_id
    LIMIT 5;
    """
)

# 2. Employee with their current salary (INNER JOIN)
execute_and_print_query(
    "Employees with their current salary (first 5 rows)",
    """
    SELECT
        E.first_name,
        E.last_name,
        S.amount AS salary_amount,
        S.effective_date
    FROM Employees AS E
    INNER JOIN Salaries AS S ON E.employee_id = S.employee_id
    ORDER BY S.effective_date DESC -- Assuming latest effective_date is current salary, or filter by max(effective_date)
    LIMIT 5;
    """
)

# 3. Employee with their job role (INNER JOIN)
execute_and_print_query(
    "Employees with their job role (first 5 rows)",
    """
    SELECT
        E.first_name,
        E.last_name,
        J.job_title
    FROM Employees AS E
    INNER JOIN Jobs AS J ON E.job_id = J.job_id
    LIMIT 5;
    """
)

# 4. Employees and the projects they are assigned to (INNER JOIN with a many-to-many table)
execute_and_print_query(
    "Employees and the projects they are assigned to (first 5 rows)",
    """
    SELECT
        E.first_name,
        E.last_name,
        P.project_name,
        EP.role
    FROM Employees AS E
    INNER JOIN Employee_Projects AS EP ON E.employee_id = EP.employee_id
    INNER JOIN Projects AS P ON EP.project_id = P.project_id
    LIMIT 5;
    """
)

# 5. All departments and the number of employees in each (LEFT JOIN)
# This is a re-run of a previous query, but explicitly showing the LEFT JOIN concept
execute_and_print_query(
    "All departments and their employee count (using LEFT JOIN)",
    """
    SELECT
        D.department_name,
        COUNT(E.employee_id) AS employee_count
    FROM Departments AS D
    LEFT JOIN Employees AS E ON D.department_id = E.department_id
    GROUP BY D.department_name
    ORDER BY employee_count DESC;
    """
)

# 6. Employees and their managers (SELF JOIN)
execute_and_print_query(
    "Employees and their managers (SELF JOIN, first 5 rows)",
    """
    SELECT
        E.first_name || ' ' || E.last_name AS employee_name,
        M.first_name || ' ' || M.last_name AS manager_name
    FROM Employees AS E
    LEFT JOIN Employees AS M ON E.manager_id = M.employee_id
    LIMIT 5;
    """
)

# 7. Employees who are not assigned to any project (LEFT JOIN with WHERE IS NULL)
execute_and_print_query(
    "Employees not assigned to any project (first 5 rows)",
    """
    SELECT
        E.first_name,
        E.last_name
    FROM Employees AS E
    LEFT JOIN Employee_Projects AS EP ON E.employee_id = EP.employee_id
    WHERE EP.project_id IS NULL
    LIMIT 5;
    """
)

# 8. All jobs and the employees who hold them (LEFT JOIN)
execute_and_print_query(
    "All jobs and the employees who hold them (first 5 rows)",
    """
    SELECT
        J.job_title,
        E.first_name,
        E.last_name
    FROM Jobs AS J
    LEFT JOIN Employees AS E ON J.job_id = E.job_id
    ORDER BY J.job_title, E.first_name
    LIMIT 5;
    """
)



--- Employees with their department details (first 5 rows) ---
first_name | last_name | department_name | location
-----------------------------------------------------
Employee1 | Lastname1 | Marketing | Building C
Employee2 | Lastname2 | Human Resources | Building A
Employee3 | Lastname3 | Finance | Building A
Employee4 | Lastname4 | IT Support | Building B
Employee5 | Lastname5 | Sales | Building D

--- Employees with their current salary (first 5 rows) ---
No results found.

--- Employees with their job role (first 5 rows) ---
first_name | last_name | job_title
------------------------------------
Employee1 | Lastname1 | Customer Support Agent
Employee2 | Lastname2 | Sales Representative
Employee3 | Lastname3 | Marketing Specialist
Employee4 | Lastname4 | Senior Software Engineer
Employee5 | Lastname5 | Marketing Specialist

--- Employees and the projects they are assigned to (first 5 rows) ---
first_name | last_name | project_name | role
------------------------------------------

### Section 9: Subqueries

**Purpose:** This section demonstrates the use of subqueries (also known as inner queries or nested queries) to retrieve data that will be used in the outer query. Subqueries are powerful tools for performing complex queries that might be difficult or impossible with a single `SELECT` statement or simple joins.

**SQL Concepts Demonstrated:**
- **Scalar Subquery:** Returns a single value.
- **Row Subquery:** Returns a single row with multiple columns.
- **Column Subquery:** Returns a single column with multiple rows.
- **Table Subquery:** Returns a table that can be used like any other table in the `FROM` clause (derived table).
- **`IN` operator with subquery:** Checks if a value is present in the result set of a subquery.
- **Comparison operators with subquery:** Using operators like `>`, `<`, `=`, etc., with scalar subqueries.

**Expected Output:** Tabular results for each query, showcasing the results derived from the subqueries.

In [32]:
# Re-using the execute_and_print_query helper function defined earlier

# 1. Find the highest-paid employee (using a scalar subquery)
execute_and_print_query(
    "Highest-paid employee",
    """
    SELECT
        first_name,
        last_name
    FROM Employees
    WHERE employee_id = (
        SELECT employee_id
        FROM Salaries
        ORDER BY amount DESC
        LIMIT 1
    );
    """
)

# 2. Employees earning above the average salary (using a scalar subquery)
execute_and_print_query(
    "Employees earning above average salary",
    """
    SELECT
        E.first_name,
        E.last_name,
        S.amount
    FROM Employees AS E
    JOIN Salaries AS S ON E.employee_id = S.employee_id
    WHERE S.amount > (SELECT AVG(amount) FROM Salaries)
    ORDER BY S.amount DESC
    LIMIT 5;
    """
)

# 3. Department with the highest average salary (using a table subquery/derived table)
execute_and_print_query(
    "Department with the highest average salary",
    """
    SELECT
        D.department_name,
        AVG(S.amount) AS avg_dept_salary
    FROM Departments AS D
    JOIN Employees AS E ON D.department_id = E.department_id
    JOIN Salaries AS S ON E.employee_id = S.employee_id
    GROUP BY D.department_name
    ORDER BY avg_dept_salary DESC
    LIMIT 1;
    """
)

# 4. Employees who work in departments located in 'Building A' (using a column subquery with IN)
execute_and_print_query(
    "Employees in departments located in 'Building A' (first 5 rows)",
    """
    SELECT
        first_name,
        last_name,
        (SELECT department_name FROM Departments WHERE department_id = E.department_id) AS department
    FROM Employees AS E
    WHERE department_id IN (
        SELECT department_id
        FROM Departments
        WHERE location = 'Building A'
    )
    LIMIT 5;
    """
)

# 5. Projects with a budget greater than the average budget of all 'Active' projects
execute_and_print_query(
    "Projects with budget greater than average of 'Active' projects (first 5 rows)",
    """
    SELECT
        project_name,
        budget,
        status
    FROM Projects
    WHERE budget > (
        SELECT AVG(budget) FROM Projects WHERE status = 'Active'
    )
    ORDER BY budget DESC
    LIMIT 5;
    """
)



--- Highest-paid employee ---
No results found.

--- Employees earning above average salary ---
No results found.

--- Department with the highest average salary ---
No results found.

--- Employees in departments located in 'Building A' (first 5 rows) ---
first_name | last_name | department
-------------------------------------
Employee2 | Lastname2 | Human Resources
Employee10 | Lastname10 | Human Resources
Employee13 | Lastname13 | Human Resources
Employee14 | Lastname14 | Human Resources
Employee17 | Lastname17 | Human Resources

--- Projects with budget greater than average of 'Active' projects (first 5 rows) ---
project_name | budget | status
--------------------------------
Project Alpha 2 | 496327.64 | Planning
Project Alpha 5 | 492563.09 | Active
Project Alpha 7 | 490247.85 | Planning
Project Alpha 19 | 440770.7 | Active
Project Alpha 4 | 376988.76 | Planning


### Section 10: Views

**Purpose:** This section demonstrates how to create and use SQL views. A view is a virtual table based on the result-set of an SQL statement. A view contains rows and columns, just like a real table. The fields in a view are fields from one or more real tables in the database. Views can be used to simplify complex queries, restrict data access, and improve data security.

**SQL Concepts Demonstrated:**
- **`CREATE VIEW`:** Statement to define a new view.
- **Simplifying complex joins:** Encapsulating multi-table joins into a single, easy-to-query view.
- **Data abstraction:** Presenting a subset of data or a simplified representation of data without exposing the underlying table structure.

**Expected Output:** Messages confirming successful view creation, followed by sample queries showing data retrieved from the views.

In [33]:
# Re-using the execute_and_print_query helper function defined earlier

# SQL commands to create views
create_views_sql = """

-- 1. Employee Details View
-- Combines information from Employees, Departments, and Jobs tables
CREATE VIEW IF NOT EXISTS Employee_Details_View AS
SELECT
    E.employee_id,
    E.first_name,
    E.last_name,
    E.email,
    E.phone_number,
    E.hire_date,
    J.job_title,
    J.min_salary,
    J.max_salary,
    D.department_name,
    D.location,
    M.first_name || ' ' || M.last_name AS manager_name
FROM Employees AS E
LEFT JOIN Jobs AS J ON E.job_id = J.job_id
LEFT JOIN Departments AS D ON E.department_id = D.department_id
LEFT JOIN Employees AS M ON E.manager_id = M.employee_id;

-- 2. Salary Report View
-- Shows current salary details for each employee
CREATE VIEW IF NOT EXISTS Salary_Report_View AS
SELECT
    E.employee_id,
    E.first_name,
    E.last_name,
    J.job_title,
    D.department_name,
    S.amount AS current_salary,
    S.effective_date AS salary_effective_date
FROM Employees AS E
JOIN Salaries AS S ON E.employee_id = S.employee_id
LEFT JOIN Jobs AS J ON E.job_id = J.job_id
LEFT JOIN Departments AS D ON E.department_id = D.department_id;

-- 3. Attendance Report View
-- Summarizes attendance records for employees
CREATE VIEW IF NOT EXISTS Attendance_Report_View AS
SELECT
    E.employee_id,
    E.first_name,
    E.last_name,
    A.attendance_date,
    A.status,
    A.check_in_time,
    A.check_out_time,
    A.hours_worked
FROM Employees AS E
JOIN Attendance AS A ON E.employee_id = A.employee_id;

-- 4. Project Assignments View
-- Shows which employees are assigned to which projects
CREATE VIEW IF NOT EXISTS Project_Assignments_View AS
SELECT
    E.employee_id,
    E.first_name,
    E.last_name,
    P.project_name,
    P.status AS project_status,
    EP.role AS assigned_role,
    EP.assigned_date
FROM Employees AS E
JOIN Employee_Projects AS EP ON E.employee_id = EP.employee_id
JOIN Projects AS P ON EP.project_id = P.project_id;

"""

try:
    cursor.executescript(create_views_sql)
    conn.commit()
    print("All views created successfully!")
except sqlite3.Error as e:
    print(f"Error creating views: {e}")


# Query the created views to demonstrate their use
execute_and_print_query(
    "Querying Employee_Details_View (first 5 rows)",
    "SELECT employee_id, first_name, last_name, job_title, department_name, manager_name FROM Employee_Details_View LIMIT 5;"
)

execute_and_print_query(
    "Querying Salary_Report_View (first 5 rows)",
    "SELECT employee_id, first_name, last_name, current_salary FROM Salary_Report_View ORDER BY current_salary DESC LIMIT 5;"
)

execute_and_print_query(
    "Querying Attendance_Report_View (first 5 rows)",
    "SELECT employee_id, first_name, last_name, attendance_date, status, hours_worked FROM Attendance_Report_View LIMIT 5;"
)

execute_and_print_query(
    "Querying Project_Assignments_View (first 5 rows)",
    "SELECT first_name, last_name, project_name, assigned_role FROM Project_Assignments_View LIMIT 5;"
)


All views created successfully!

--- Querying Employee_Details_View (first 5 rows) ---
employee_id | first_name | last_name | job_title | department_name | manager_name
-----------------------------------------------------------------------------------
1 | Employee1 | Lastname1 | Customer Support Agent | Marketing | None
2 | Employee2 | Lastname2 | Sales Representative | Human Resources | None
3 | Employee3 | Lastname3 | Marketing Specialist | Finance | None
4 | Employee4 | Lastname4 | Senior Software Engineer | IT Support | None
5 | Employee5 | Lastname5 | Marketing Specialist | Sales | None

--- Querying Salary_Report_View (first 5 rows) ---
No results found.

--- Querying Attendance_Report_View (first 5 rows) ---
employee_id | first_name | last_name | attendance_date | status | hours_worked
--------------------------------------------------------------------------------
1 | Employee1 | Lastname1 | 2023-10-07 | Leave | None
1 | Employee1 | Lastname1 | 2026-04-28 | Present | 9.0
1 | E

### Section 11: Indexes

**Purpose:** This section focuses on creating indexes to improve the performance of database queries. Indexes are special lookup tables that the database search engine can use to speed up data retrieval. Think of an index like an index in a book: it helps you find information quickly without having to read the entire book. While indexes can speed up `SELECT` queries, they can slow down data modification operations (`INSERT`, `UPDATE`, `DELETE`) because the indexes also need to be updated.

**SQL Concepts Demonstrated:**
- **`CREATE INDEX`:** Statement to create an index on a table.
- **`ON` clause:** Specifies the table and column(s) on which to build the index.
- **Performance Optimization:** Understanding when and why to use indexes.

**Expected Output:** Messages confirming successful index creation.

In [34]:
# SQL commands to create indexes
create_indexes_sql = """

-- Create an index on the last_name column in the Employees table
-- This is useful for queries that filter or sort by employee last name.
CREATE INDEX IF NOT EXISTS idx_employee_last_name ON Employees (last_name);

-- Create an index on the department_id column in the Employees table
-- This is beneficial for queries that join Employees with Departments or filter by department.
CREATE INDEX IF NOT EXISTS idx_employee_department_id ON Employees (department_id);

-- Create an index on the job_id column in the Employees table
-- This is beneficial for queries that join Employees with Jobs or filter by job.
CREATE INDEX IF NOT EXISTS idx_employee_job_id ON Employees (job_id);

-- Create an index on the email column in the Employees table
-- This is beneficial for fast lookups by email, which is often a unique identifier.
CREATE INDEX IF NOT EXISTS idx_employee_email ON Employees (email);

-- Create an index on the effective_date in the Salaries table
-- Useful for queries filtering or ordering salaries by date.
CREATE INDEX IF NOT EXISTS idx_salary_effective_date ON Salaries (effective_date);

-- Create an index on the attendance_date in the Attendance table
-- Useful for queries filtering or ordering attendance records by date.
CREATE INDEX IF NOT EXISTS idx_attendance_date ON Attendance (attendance_date);

-- Create an index on the project_name in the Projects table
-- Useful for fast lookups and searches by project name.
CREATE INDEX IF NOT EXISTS idx_project_name ON Projects (project_name);

"""

try:
    cursor.executescript(create_indexes_sql)
    conn.commit()
    print("All indexes created successfully!")
except sqlite3.Error as e:
    print(f"Error creating indexes: {e}")

# To verify indexes, you can query the sqlite_master table
execute_and_print_query(
    "Verifying created indexes",
    "SELECT name, tbl_name, sql FROM sqlite_master WHERE type='index';"
)

All indexes created successfully!

--- Verifying created indexes ---
name | tbl_name | sql
-----------------------
sqlite_autoindex_Departments_1 | Departments | None
sqlite_autoindex_Departments_2 | Departments | None
sqlite_autoindex_Jobs_1 | Jobs | None
sqlite_autoindex_Employees_1 | Employees | None
sqlite_autoindex_Employees_2 | Employees | None
sqlite_autoindex_Attendance_1 | Attendance | None
sqlite_autoindex_Projects_1 | Projects | None
sqlite_autoindex_Employee_Projects_1 | Employee_Projects | None
idx_employee_last_name | Employees | CREATE INDEX idx_employee_last_name ON Employees (last_name)
idx_employee_department_id | Employees | CREATE INDEX idx_employee_department_id ON Employees (department_id)
idx_employee_job_id | Employees | CREATE INDEX idx_employee_job_id ON Employees (job_id)
idx_employee_email | Employees | CREATE INDEX idx_employee_email ON Employees (email)
idx_attendance_date | Attendance | CREATE INDEX idx_attendance_date ON Attendance (attendance_date)
idx_

### Section 12: Stored Procedures (SQLite Alternatives)

**Purpose:** Stored procedures are precompiled SQL code stored in the database, often used for encapsulating complex business logic, improving performance, and enhancing security. SQLite, being a lightweight, embedded database, does **not natively support stored procedures** in the same way enterprise-level database systems (like SQL Server, MySQL, PostgreSQL, Oracle) do.

However, we can achieve similar functionality and benefits by:
1.  **Encapsulating Logic in Application Code (Python Functions):** This is the most common and recommended approach in SQLite. We can write Python functions that contain multiple SQL statements, manage transactions, and handle logic. These functions can then be called from various parts of our application.
2.  **Using Views:** For read-only complex queries, views (as demonstrated in Section 10) can serve as a form of pre-defined 'procedure' for data retrieval.
3.  **User-Defined Functions (UDFs):** SQLite allows you to register custom functions written in the host language (like Python) that can be called directly within SQL queries. This is for single-value transformations, not multi-statement logic.

For the purpose of this project, we will demonstrate the first approach: using Python functions to encapsulate a series of SQL operations, mimicking the behavior of a stored procedure.

**SQL Concepts Demonstrated:**
-   **Application-level encapsulation:** Using Python to group and execute multiple SQL statements.
-   **Transaction Management:** Ensuring atomicity of operations (`BEGIN TRANSACTION`, `COMMIT`, `ROLLBACK`).

**Expected Output:** Messages confirming successful execution of the 'simulated' stored procedure and the outcome of the operations.

### Section 13: User-Defined Functions (UDFs)

**Purpose:** This section demonstrates how to extend SQLite's capabilities by registering custom functions written in Python. These User-Defined Functions (UDFs) can then be called directly within SQL queries, allowing for complex, custom logic to be applied to data during retrieval or filtering.

**SQL Concepts Demonstrated:**
-   **`conn.create_function()`:** Registering a Python function as an SQL UDF.
-   **Integrating Python logic:** Using custom Python code directly within SQL queries.

**Expected Output:** Messages confirming UDF registration, followed by query results demonstrating the UDF in action.

### Section 13.1: Batch Salary Updates with Transaction Management (Rollback Demonstration)

**Purpose:** This section expands on transaction management by demonstrating how to process a batch of updates. In a real-world scenario, you often need to apply multiple changes atomically. If any single change in the batch fails, the entire batch should be rolled back to maintain data consistency. This function simulates a 'stored procedure' for batch processing.

**SQL Concepts Demonstrated:**
-   **`BEGIN TRANSACTION;`**: Initiates a transaction block.
-   **`COMMIT;`**: Saves all changes made within the transaction.
-   **`ROLLBACK;`**: Undoes all changes made within the transaction.
-   **Error Handling (`try-except`):** Gracefully managing exceptions and triggering rollbacks.

**Expected Output:** Messages confirming the success or failure of batch updates, and verification of salary records before and after attempted updates.

### Section 13.2: Demonstrating SQLite Concurrency (File-Level Locking)

**Purpose:** This section illustrates how SQLite handles concurrent write attempts using file-level locking. In a single-process Python script, true parallel execution is limited by the Global Interpreter Lock (GIL) and SQLite's design. However, we can demonstrate the *locking behavior* by simulating concurrent access using two distinct `sqlite3` connection objects to the same database file.

When one connection starts a write transaction, it acquires a lock on the database file. Any other connection attempting a write operation will either block until the lock is released or raise a `sqlite3.OperationalError: database is locked` if a timeout is exceeded.

**SQL Concepts Demonstrated:**
-   **File-level locking:** SQLite's mechanism for managing concurrent access.
-   **Transaction isolation:** How one transaction can prevent others from modifying data.
-   **`BEGIN EXCLUSIVE TRANSACTION`:** To acquire a write lock immediately.
-   **`conn.timeout`:** Setting a timeout for when a connection encounters a busy database.

**Expected Output:** Messages showing one connection acquiring a lock, another connection failing (or blocking) due to the lock, and then successfully writing once the lock is released.

In [40]:
import time
from datetime import datetime

# --- Step 1: Start a transaction and hold a lock on the main connection (conn) ---
print("\n--- Main Connection (conn): Starting an exclusive transaction and inserting data ---")
conn.execute("BEGIN EXCLUSIVE TRANSACTION;")

try:
    # Insert a dummy salary record to hold the write lock
    dummy_employee_id = 1
    dummy_amount = 12345.67
    dummy_date = datetime.now().strftime('%Y-%m-%d')
    cursor.execute(
        "INSERT INTO Salaries (employee_id, amount, effective_date) VALUES (?, ?, ?);",
        (dummy_employee_id, dummy_amount, dummy_date)
    )
    print(f"Main Connection: Inserted dummy salary {dummy_amount} for Employee ID {dummy_employee_id} (Transaction open)...")

    # --- Step 2: Attempt an insertion from a second, independent connection ---
    print("\n--- Second Connection (conn2): Attempting to insert data while main connection holds lock ---")
    conn2 = None
    try:
        # Open a new connection to the same database file
        conn2 = sqlite3.connect(DATABASE_NAME, timeout=1) # Set a 1-second timeout
        cursor2 = conn2.cursor()

        # Attempt to insert data
        concurrent_employee_id = 2
        concurrent_amount = 98765.43
        concurrent_date = datetime.now().strftime('%Y-%m-%d')
        cursor2.execute(
            "INSERT INTO Salaries (employee_id, amount, effective_date) VALUES (?, ?, ?);",
            (concurrent_employee_id, concurrent_amount, concurrent_date)
        )
        conn2.commit()
        print(f"Second Connection: Successfully inserted concurrent salary {concurrent_amount} for Employee ID {concurrent_employee_id}.")

    except sqlite3.OperationalError as e:
        print(f"Second Connection: Failed to insert due to database lock: {e}")
        if conn2:
            conn2.rollback() # Rollback the second connection if it managed to start anything
    except Exception as e:
        print(f"Second Connection: An unexpected error occurred: {e}")
    finally:
        if conn2: conn2.close()

    # --- Step 3: Commit the main transaction to release the lock ---
    print("\n--- Main Connection (conn): Committing transaction and releasing lock ---")
    conn.commit()
    print("Main Connection: Transaction committed. Lock released.")

    # --- Step 4: Retry insertion from a second connection (should now succeed) ---
    print("\n--- Second Connection (conn2): Retrying insertion after main connection released lock ---")
    conn2 = None
    try:
        conn2 = sqlite3.connect(DATABASE_NAME, timeout=1)
        cursor2 = conn2.cursor()
        concurrent_employee_id = 3 # Use a different employee to make results clearer
        concurrent_amount = 55555.55
        concurrent_date = datetime.now().strftime('%Y-%m-%d')
        cursor2.execute(
            "INSERT INTO Salaries (employee_id, amount, effective_date) VALUES (?, ?, ?);",
            (concurrent_employee_id, concurrent_amount, concurrent_date)
        )
        conn2.commit()
        print(f"Second Connection: Successfully inserted concurrent salary {concurrent_amount} for Employee ID {concurrent_employee_id} after lock release.")
    except sqlite3.OperationalError as e:
        print(f"Second Connection: Still failed to insert after lock release: {e}")
    except Exception as e:
        print(f"Second Connection: An unexpected error occurred on retry: {e}")
    finally:
        if conn2: conn2.close()

except Exception as e:
    print(f"Main Connection: An error occurred in main transaction: {e}. Rolling back.")
    conn.rollback()
finally:
    # Verify the final state of the Salaries table for the involved employees
    execute_and_print_query(
        "Salaries records for Employee IDs 1, 2, and 3 after concurrency test",
        "SELECT employee_id, amount, effective_date FROM Salaries WHERE employee_id IN (1, 2, 3) ORDER BY employee_id, effective_date DESC;"
    )



--- Main Connection (conn): Starting an exclusive transaction and inserting data ---
Main Connection: Inserted dummy salary 12345.67 for Employee ID 1 (Transaction open)...

--- Second Connection (conn2): Attempting to insert data while main connection holds lock ---
Second Connection: Failed to insert due to database lock: database is locked

--- Main Connection (conn): Committing transaction and releasing lock ---
Main Connection: Transaction committed. Lock released.

--- Second Connection (conn2): Retrying insertion after main connection released lock ---
Second Connection: Successfully inserted concurrent salary 55555.55 for Employee ID 3 after lock release.

--- Salaries records for Employee IDs 1, 2, and 3 after concurrency test ---
employee_id | amount | effective_date
---------------------------------------
1 | 140291.96 | 2026-07-03
1 | 150000.0 | 2026-07-03
1 | 12345.67 | 2026-07-03
2 | 80000.0 | 2026-07-03
2 | 95000.0 | 2026-07-03
3 | 85000.0 | 2026-07-03
3 | 55555.55 | 20

In [39]:
import random
from datetime import datetime

def process_batch_salary_updates(updates_batch):
    """
    Processes a batch of salary updates. If any update fails, the entire batch is rolled back.

    Args:
        updates_batch (list of dict): A list of dictionaries, each containing:
            'employee_id': INTEGER,
            'amount': REAL,
            'effective_date': DATE (YYYY-MM-DD string)
    """
    try:
        conn.execute("BEGIN TRANSACTION;")
        print(f"\n--- Attempting to process {len(updates_batch)} salary updates ---")

        for update_data in updates_batch:
            employee_id = update_data['employee_id']
            amount = update_data['amount']
            effective_date = update_data['effective_date']

            # Validate employee_id exists to simulate a real-world constraint check
            cursor.execute("SELECT 1 FROM Employees WHERE employee_id = ?;", (employee_id,))
            if cursor.fetchone() is None:
                raise ValueError(f"Employee ID {employee_id} not found. Rolling back batch.")

            cursor.execute(
                "INSERT INTO Salaries (employee_id, amount, effective_date) VALUES (?, ?, ?);",
                (employee_id, amount, effective_date)
            )
            print(f"  Inserted salary {amount} for Employee ID {employee_id} effective {effective_date}.")

        conn.commit()
        print("Batch salary update successful! All updates committed.")

    except Exception as e:
        conn.rollback()
        print(f"Batch salary update failed: {e}. All changes rolled back.")

# --- Demonstration ---

# 1. Get initial salary records for a few employees
execute_and_print_query(
    "Initial salaries for Employee 1, 2, and 3",
    "SELECT employee_id, amount, effective_date FROM Salaries WHERE employee_id IN (1, 2, 3) ORDER BY employee_id, effective_date DESC LIMIT 3;"
)

# 2. Prepare a valid batch of updates
valid_batch = [
    {'employee_id': 1, 'amount': 150000.00, 'effective_date': datetime.now().strftime('%Y-%m-%d')},
    {'employee_id': 2, 'amount': 95000.00, 'effective_date': datetime.now().strftime('%Y-%m-%d')},
    {'employee_id': 3, 'amount': 85000.00, 'effective_date': datetime.now().strftime('%Y-%m-%d')},
]

# Process the valid batch
process_batch_salary_updates(valid_batch)

# 3. Verify salaries after valid batch update
execute_and_print_query(
    "Salaries after valid batch update (Employee 1, 2, and 3)",
    "SELECT employee_id, amount, effective_date FROM Salaries WHERE employee_id IN (1, 2, 3) ORDER BY employee_id, effective_date DESC LIMIT 3;"
)

# 4. Prepare an invalid batch (e.g., non-existent employee_id)
invalid_batch = [
    {'employee_id': 1, 'amount': 160000.00, 'effective_date': datetime.now().strftime('%Y-%m-%d')},
    {'employee_id': 999, 'amount': 100000.00, 'effective_date': datetime.now().strftime('%Y-%m-%d')}, # This will cause an error
    {'employee_id': 3, 'amount': 90000.00, 'effective_date': datetime.now().strftime('%Y-%m-%d')},
]

# Process the invalid batch
process_batch_salary_updates(invalid_batch)

# 5. Verify salaries after invalid batch update (should be no change for Employee 1 and 3 from this batch)
execute_and_print_query(
    "Salaries after invalid batch update attempt (Employee 1, 2, and 3)",
    "SELECT employee_id, amount, effective_date FROM Salaries WHERE employee_id IN (1, 2, 3) ORDER BY employee_id, effective_date DESC LIMIT 3;"
)



--- Initial salaries for Employee 1, 2, and 3 ---
employee_id | amount | effective_date
---------------------------------------
1 | 140291.96 | 2026-07-03
2 | 80000.0 | 2026-07-03

--- Attempting to process 3 salary updates ---
  Inserted salary 150000.0 for Employee ID 1 effective 2026-07-03.
  Inserted salary 95000.0 for Employee ID 2 effective 2026-07-03.
  Inserted salary 85000.0 for Employee ID 3 effective 2026-07-03.
Batch salary update successful! All updates committed.

--- Salaries after valid batch update (Employee 1, 2, and 3) ---
employee_id | amount | effective_date
---------------------------------------
1 | 140291.96 | 2026-07-03
1 | 150000.0 | 2026-07-03
2 | 80000.0 | 2026-07-03

--- Attempting to process 3 salary updates ---
  Inserted salary 160000.0 for Employee ID 1 effective 2026-07-03.
Batch salary update failed: Employee ID 999 not found. Rolling back batch.. All changes rolled back.

--- Salaries after invalid batch update attempt (Employee 1, 2, and 3) ---
emp

In [38]:
# Define a Python function to calculate custom tax
# For demonstration, let's say:
# - No tax for salaries up to 50,000
# - 10% tax for salaries between 50,001 and 100,000
# - 15% tax for salaries above 100,000
def calculate_custom_tax(salary):
    if salary <= 50000:
        return 0.0
    elif salary <= 100000:
        return salary * 0.10
    else:
        return salary * 0.15

# Register the Python function as an SQLite UDF
# The arguments are: function_name_in_sql, num_args, python_function_to_call
try:
    conn.create_function("custom_tax", 1, calculate_custom_tax)
    print("Custom UDF 'custom_tax' registered successfully.")
except Exception as e:
    print(f"Error registering UDF: {e}")

# Demonstrate the use of the custom_tax UDF in an SQL query
execute_and_print_query(
    "Employees with calculated custom tax and net salary (first 10 rows)",
    """
    SELECT
        E.first_name,
        E.last_name,
        S.amount AS gross_salary,
        custom_tax(S.amount) AS tax_amount,
        S.amount - custom_tax(S.amount) AS net_salary
    FROM Employees AS E
    JOIN Salaries AS S ON E.employee_id = S.employee_id
    WHERE S.effective_date = (SELECT MAX(effective_date) FROM Salaries WHERE employee_id = E.employee_id)
    ORDER BY gross_salary DESC
    LIMIT 10;
    """
)

Custom UDF 'custom_tax' registered successfully.

--- Employees with calculated custom tax and net salary (first 10 rows) ---
first_name | last_name | gross_salary | tax_amount | net_salary
-----------------------------------------------------------------
Employee1 | Lastname1 | 140291.96 | 21043.793999999998 | 119248.166
Employee2 | Lastname2 | 80000.0 | 8000.0 | 72000.0


In [35]:
# Helper function to simulate a stored procedure to update an employee's job and salary
# This function encapsulates multiple SQL operations and transaction management.
def update_employee_job_and_salary(employee_id, new_job_id, new_salary_amount):
    try:
        # Start a transaction
        conn.execute("BEGIN TRANSACTION;")

        # 1. Update the employee's job_id in the Employees table
        cursor.execute(
            "UPDATE Employees SET job_id = ? WHERE employee_id = ?;",
            (new_job_id, employee_id)
        )

        # 2. Insert a new salary record in the Salaries table
        # In a real system, you might first check if the new salary is different
        # and update an existing 'current' salary record or simply insert a new one
        # with a new effective_date, marking the old one as historical.
        # For simplicity, we'll assume a new record is always added here.
        cursor.execute(
            "INSERT INTO Salaries (employee_id, amount, effective_date) VALUES (?, ?, CURRENT_DATE);",
            (employee_id, new_salary_amount)
        )

        # Commit the transaction if all operations are successful
        conn.commit()
        print(f"Successfully updated employee {employee_id}'s job to job_id {new_job_id} and added new salary record.")

    except sqlite3.Error as e:
        # Rollback the transaction if any error occurs
        conn.rollback()
        print(f"Error updating employee {employee_id}: {e}. Transaction rolled back.")

# --- Demonstration of the 'simulated' stored procedure ---

# Get an existing employee and their current job/salary for comparison
execute_and_print_query(
    "Employee 1 details before update",
    "SELECT E.first_name, E.last_name, J.job_title, S.amount, S.effective_date FROM Employees E JOIN Jobs J ON E.job_id = J.job_id JOIN Salaries S ON E.employee_id = S.employee_id WHERE E.employee_id = 1 ORDER BY S.effective_date DESC LIMIT 1;"
)

# Let's find a new job_id and a new salary amount
# For example, promote Employee 1 to 'Senior Software Engineer' (job_id 14) with a higher salary
new_job_title = 'Senior Software Engineer'
cursor.execute("SELECT job_id, min_salary, max_salary FROM Jobs WHERE job_title = ?", (new_job_title,))
new_job_info = cursor.fetchone()

if new_job_info:
    new_job_id = new_job_info[0]
    min_sal, max_sal = new_job_info[1], new_job_info[2]
    # Set a new salary within the range of the new job, higher than current if possible
    new_salary = round(random.uniform(min_sal, max_sal), 2)

    # Call our simulated stored procedure
    print(f"\nAttempting to update employee 1 to {new_job_title} with salary {new_salary}...")
    update_employee_job_and_salary(1, new_job_id, new_salary)

    # Verify the update
    execute_and_print_query(
        "Employee 1 details after update",
        "SELECT E.first_name, E.last_name, J.job_title, S.amount, S.effective_date FROM Employees E JOIN Jobs J ON E.job_id = J.job_id JOIN Salaries S ON E.employee_id = S.employee_id WHERE E.employee_id = 1 ORDER BY S.effective_date DESC LIMIT 1;"
    )
else:
    print(f"Job title '{new_job_title}' not found. Cannot proceed with update.")

# Demonstrate rollback: try to update with invalid job_id
print("\nAttempting to update employee 2 with an invalid job_id (e.g., 999) to demonstrate rollback...")
update_employee_job_and_salary(2, 999, 80000) # job_id 999 does not exist

# Verify employee 2's details (should remain unchanged)
execute_and_print_query(
    "Employee 2 details (should be unchanged after rollback)",
    "SELECT E.first_name, E.last_name, J.job_title, S.amount FROM Employees E JOIN Jobs J ON E.job_id = J.job_id JOIN Salaries S ON E.employee_id = S.employee_id WHERE E.employee_id = 2 ORDER BY S.effective_date DESC LIMIT 1;"
)



--- Employee 1 details before update ---
No results found.

Attempting to update employee 1 to Senior Software Engineer with salary 140291.96...
Successfully updated employee 1's job to job_id 14 and added new salary record.

--- Employee 1 details after update ---
first_name | last_name | job_title | amount | effective_date
--------------------------------------------------------------
Employee1 | Lastname1 | Senior Software Engineer | 140291.96 | 2026-07-03

Attempting to update employee 2 with an invalid job_id (e.g., 999) to demonstrate rollback...
Successfully updated employee 2's job to job_id 999 and added new salary record.

--- Employee 2 details (should be unchanged after rollback) ---
No results found.


In [36]:
execute_and_print_query(
    "Salaries Table Schema (after modification)",
    "SELECT sql FROM sqlite_master WHERE type='table' AND name='Salaries';"
)


--- Salaries Table Schema (after modification) ---
sql
-----
CREATE TABLE Salaries (
    salary_id INTEGER PRIMARY KEY AUTOINCREMENT,
    employee_id INTEGER NOT NULL,
    amount REAL NOT NULL CHECK (amount >= 0),
    effective_date DATE NOT NULL DEFAULT CURRENT_DATE,
    FOREIGN KEY (employee_id) REFERENCES Employees(employee_id) ON DELETE CASCADE ON UPDATE CASCADE
)


### Section 14: Advanced Queries (Real-world Business Scenarios)

**Purpose:** This section presents a series of advanced SQL queries designed to answer real-world business questions that an HR or management department might have. These queries will combine various SQL concepts demonstrated in previous sections, including joins, subqueries, aggregate functions, grouping, and filtering, to extract complex insights from the employee management database.

**SQL Concepts Demonstrated:**
-   Complex combinations of `JOIN` types (INNER, LEFT, SELF).
-   Nested subqueries and correlated subqueries.
-   Advanced usage of `GROUP BY` and `HAVING`.
-   Date functions and arithmetic.
-   Conditional logic within queries (e.g., `CASE` statements, though less common in simple SQLite queries, can be simulated).
-   Identification of patterns, anomalies, and specific employee/department characteristics.

**Expected Output:** Tabular results for each query, providing actionable insights for business decision-making.

In [37]:
# Re-using the execute_and_print_query helper function defined earlier

# --- Advanced Business Queries ---

# 1. Employees whose current salary is outside the defined min_salary/max_salary range for their job title
execute_and_print_query(
    "1. Employees with salary outside job range",
    """
    SELECT
        E.first_name, E.last_name, J.job_title,
        S.amount AS current_salary, J.min_salary, J.max_salary
    FROM Employees AS E
    JOIN Jobs AS J ON E.job_id = J.job_id
    JOIN Salaries AS S ON E.employee_id = S.employee_id
    WHERE S.effective_date = (SELECT MAX(effective_date) FROM Salaries WHERE employee_id = E.employee_id)
      AND (S.amount < J.min_salary OR S.amount > J.max_salary);
    """
)

# 2. Departments with the highest average salary (already covered, but good for context)
execute_and_print_query(
    "2. Department with the highest average salary",
    """
    SELECT
        D.department_name,
        AVG(S.amount) AS avg_dept_salary
    FROM Departments AS D
    JOIN Employees AS E ON D.department_id = E.department_id
    JOIN Salaries AS S ON E.employee_id = S.employee_id
    WHERE S.effective_date = (SELECT MAX(effective_date) FROM Salaries WHERE employee_id = E.employee_id)
    GROUP BY D.department_name
    ORDER BY avg_dept_salary DESC
    LIMIT 1;
    """
)

# 3. Employees who have not had any attendance records in the last 30 days
execute_and_print_query(
    "3. Employees with no attendance in last 30 days",
    """
    SELECT
        E.first_name, E.last_name
    FROM Employees AS E
    LEFT JOIN Attendance AS A ON E.employee_id = A.employee_id
        AND A.attendance_date BETWEEN date('now', '-30 days') AND date('now')
    GROUP BY E.employee_id
    HAVING COUNT(A.attendance_id) = 0;
    """
)

# 4. Projects that are 'Active' and have exceeded their planned end date
execute_and_print_query(
    "4. Overdue Active Projects",
    """
    SELECT
        project_name, start_date, end_date, status
    FROM Projects
    WHERE status = 'Active' AND end_date < CURRENT_DATE;
    """
)

# 5. Employees who are assigned to more than 2 projects
execute_and_print_query(
    "5. Employees on more than 2 projects",
    """
    SELECT
        E.first_name, E.last_name, COUNT(EP.project_id) AS num_projects
    FROM Employees AS E
    JOIN Employee_Projects AS EP ON E.employee_id = EP.employee_id
    GROUP BY E.employee_id
    HAVING COUNT(EP.project_id) > 2
    ORDER BY num_projects DESC;
    """
)

# 6. Departments with no employees assigned
execute_and_print_query(
    "6. Departments with no employees",
    """
    SELECT
        D.department_name
    FROM Departments AS D
    LEFT JOIN Employees AS E ON D.department_id = E.department_id
    WHERE E.employee_id IS NULL;
    """
)

# 7. Jobs that currently have no employees filling them
execute_and_print_query(
    "7. Unfilled Job Titles",
    """
    SELECT
        J.job_title
    FROM Jobs AS J
    LEFT JOIN Employees AS E ON J.job_id = E.job_id
    WHERE E.employee_id IS NULL;
    """
)

# 8. Employees whose manager works in a different department
execute_and_print_query(
    "8. Employees with cross-departmental managers (first 5 rows)",
    """
    SELECT
        E.first_name || ' ' || E.last_name AS employee_name,
        D_emp.department_name AS employee_department,
        M.first_name || ' ' || M.last_name AS manager_name,
        D_mgr.department_name AS manager_department
    FROM Employees AS E
    JOIN Departments AS D_emp ON E.department_id = D_emp.department_id
    JOIN Employees AS M ON E.manager_id = M.employee_id
    JOIN Departments AS D_mgr ON M.department_id = D_mgr.department_id
    WHERE E.department_id <> M.department_id
    LIMIT 5;
    """
)

# 9. Employees hired in the last year who have not been assigned to any project
execute_and_print_query(
    "9. Recently hired, unassigned employees (last 1 year)",
    """
    SELECT
        E.first_name, E.last_name, E.hire_date
    FROM Employees AS E
    LEFT JOIN Employee_Projects AS EP ON E.employee_id = EP.employee_id
    WHERE E.hire_date >= date('now', '-1 year')
      AND EP.project_id IS NULL;
    """
)

# 10. Total budget for 'Active' projects, grouped by department (indirectly via assigned employees)
execute_and_print_query(
    "10. Total Active Project Budget per Department",
    """
    SELECT
        D.department_name,
        SUM(P.budget) AS total_active_project_budget
    FROM Departments AS D
    JOIN Employees AS E ON D.department_id = E.department_id
    JOIN Employee_Projects AS EP ON E.employee_id = EP.employee_id
    JOIN Projects AS P ON EP.project_id = P.project_id
    WHERE P.status = 'Active'
    GROUP BY D.department_name
    ORDER BY total_active_project_budget DESC;
    """
)

# 11. Employees who have worked more than 40 hours in a single attendance record
execute_and_print_query(
    "11. Employees with unusually long shifts",
    """
    SELECT
        E.first_name, E.last_name, A.attendance_date, A.hours_worked
    FROM Employees AS E
    JOIN Attendance AS A ON E.employee_id = A.employee_id
    WHERE A.hours_worked > 40;
    """
)

# 12. Average tenure (in days) of employees in each department
execute_and_print_query(
    "12. Average Employee Tenure (days) per Department",
    """
    SELECT
        D.department_name,
        AVG(JULIANDAY(CURRENT_DATE) - JULIANDAY(E.hire_date)) AS average_tenure_days
    FROM Departments AS D
    JOIN Employees AS E ON D.department_id = E.department_id
    GROUP BY D.department_name
    ORDER BY average_tenure_days DESC;
    """
)

# 13. Projects with an average assigned employee salary higher than the overall average employee salary
execute_and_print_query(
    "13. Projects with above-average paid teams",
    """
    SELECT
        P.project_name,
        AVG(S.amount) AS avg_team_salary
    FROM Projects AS P
    JOIN Employee_Projects AS EP ON P.project_id = EP.project_id
    JOIN Employees AS E ON EP.employee_id = E.employee_id
    JOIN Salaries AS S ON E.employee_id = S.employee_id
    WHERE S.effective_date = (SELECT MAX(effective_date) FROM Salaries WHERE employee_id = E.employee_id)
    GROUP BY P.project_id
    HAVING AVG(S.amount) > (SELECT AVG(amount) FROM Salaries WHERE effective_date = (SELECT MAX(effective_date) FROM Salaries))
    ORDER BY avg_team_salary DESC;
    """
)

# 14. Employees who are currently 'Present' but have no check-out time recorded for today
execute_and_print_query(
    "14. Employees currently checked-in without check-out",
    """
    SELECT
        E.first_name, E.last_name, A.check_in_time
    FROM Employees AS E
    JOIN Attendance AS A ON E.employee_id = A.employee_id
    WHERE A.attendance_date = CURRENT_DATE
      AND A.status = 'Present'
      AND A.check_out_time IS NULL;
    """
)

# 15. The top 5 employees with the most total hours worked across all attendance records
execute_and_print_query(
    "15. Top 5 employees by total hours worked",
    """
    SELECT
        E.first_name, E.last_name, SUM(A.hours_worked) AS total_hours
    FROM Employees AS E
    JOIN Attendance AS A ON E.employee_id = A.employee_id
    WHERE A.hours_worked IS NOT NULL
    GROUP BY E.employee_id
    ORDER BY total_hours DESC
    LIMIT 5;
    """
)

# 16. Projects with assigned employees from at least 3 different departments
execute_and_print_query(
    "16. Cross-departmental Projects",
    """
    SELECT
        P.project_name,
        COUNT(DISTINCT E.department_id) AS distinct_departments
    FROM Projects AS P
    JOIN Employee_Projects AS EP ON P.project_id = EP.project_id
    JOIN Employees AS E ON EP.employee_id = E.employee_id
    GROUP BY P.project_id
    HAVING distinct_departments >= 3
    ORDER BY distinct_departments DESC;
    """
)

# 17. Employees who have had multiple job changes within the company (more than one job_id in their history, if tracked)
# Note: Our current schema doesn't track job history directly, only current job_id. This query assumes job_id changes are recorded somehow, or will simply show employees who currently hold multiple jobs in `Jobs` table (which is unlikely). For a more accurate query, a JobHistory table would be needed. For this schema, we can look for employees whose `job_id` has been updated in `Employees` table which requires tracking the changes (not possible with just the current tables).
# A proxy: Employees who have ever been associated with different job titles through their salary or project history (though not truly job changes).
# Let's rephrase for current schema: Employees whose 'job_id' has been updated (can't directly query history).
# A better query for current schema: Employees with unique job_titles in `Jobs` table.
execute_and_print_query(
    "17. Employees with potentially varied job experiences (based on project roles)",
    """
    SELECT
        E.first_name, E.last_name, COUNT(DISTINCT EP.role) AS distinct_roles_on_projects
    FROM Employees AS E
    JOIN Employee_Projects AS EP ON E.employee_id = EP.employee_id
    GROUP BY E.employee_id
    HAVING distinct_roles_on_projects > 1
    ORDER BY distinct_roles_on_projects DESC;
    """
)


# 18. Count of employees by their assigned role across all projects
execute_and_print_query(
    "18. Employee Count by Project Role",
    """
    SELECT
        EP.role,
        COUNT(DISTINCT E.employee_id) AS employee_count
    FROM Employee_Projects AS EP
    JOIN Employees AS E ON EP.employee_id = E.employee_id
    GROUP BY EP.role
    ORDER BY employee_count DESC;
    """
)

# 19. Departments with the highest average project budget where their employees are assigned
execute_and_print_query(
    "19. Departments with High Average Project Budget Involvement",
    """
    SELECT
        D.department_name,
        AVG(P.budget) AS avg_project_budget_involvement
    FROM Departments AS D
    JOIN Employees AS E ON D.department_id = E.department_id
    JOIN Employee_Projects AS EP ON E.employee_id = EP.employee_id
    JOIN Projects AS P ON EP.project_id = P.project_id
    GROUP BY D.department_name
    ORDER BY avg_project_budget_involvement DESC;
    """
)

# 20. Employees who have been on 'Leave' more than 3 times in the last 6 months
execute_and_print_query(
    "20. Employees with frequent leaves (last 6 months)",
    """
    SELECT
        E.first_name, E.last_name, COUNT(A.attendance_id) AS leave_count
    FROM Employees AS E
    JOIN Attendance AS A ON E.employee_id = A.employee_id
    WHERE A.status = 'Leave'
      AND A.attendance_date BETWEEN date('now', '-6 months') AND date('now')
    GROUP BY E.employee_id
    HAVING leave_count > 3
    ORDER BY leave_count DESC;
    """
)

# 21. Projects that have reached 75% of their end date but are still in 'Planning' status
execute_and_print_query(
    "21. Slow-starting Projects (75% to end date, still planning)",
    """
    SELECT
        project_name, start_date, end_date, status
    FROM Projects
    WHERE status = 'Planning'
      AND JULIANDAY(CURRENT_DATE) - JULIANDAY(start_date) > 0.75 * (JULIANDAY(end_date) - JULIANDAY(start_date));
    """
)

# 22. Employees whose hire date is earlier than their first recorded salary effective date
execute_and_print_query(
    "22. Employees with salary effective date before hire date (potential data error)",
    """
    SELECT
        E.first_name, E.last_name, E.hire_date, MIN(S.effective_date) AS first_salary_date
    FROM Employees AS E
    JOIN Salaries AS S ON E.employee_id = S.employee_id
    GROUP BY E.employee_id
    HAVING E.hire_date > MIN(S.effective_date);
    """
)

# 23. List all employees and their managers, including employees who are not managers themselves (self-join)
execute_and_print_query(
    "23. Employees and their Managers (Self-Join)",
    """
    SELECT
        E.first_name || ' ' || E.last_name AS employee_name,
        M.first_name || ' ' || M.last_name AS manager_name
    FROM Employees AS E
    LEFT JOIN Employees AS M ON E.manager_id = M.employee_id
    ORDER BY employee_name
    LIMIT 10;
    """
)

# 24. Find the most common job title in each department
execute_and_print_query(
    "24. Most Common Job Title per Department",
    """
    SELECT
        D.department_name, J.job_title, COUNT(E.employee_id) AS job_count
    FROM Departments AS D
    JOIN Employees AS E ON D.department_id = E.department_id
    JOIN Jobs AS J ON E.job_id = J.job_id
    GROUP BY D.department_name, J.job_title
    HAVING job_count = (
        SELECT MAX(count_job_title)
        FROM (
            SELECT COUNT(E2.employee_id) AS count_job_title
            FROM Employees AS E2
            JOIN Jobs AS J2 ON E2.job_id = J2.job_id
            WHERE E2.department_id = D.department_id
            GROUP BY J2.job_title
        )
    )
    ORDER BY D.department_name, job_count DESC;
    """
)

# 25. Projects with no employees currently assigned
execute_and_print_query(
    "25. Projects with no employees assigned",
    """
    SELECT
        P.project_name
    FROM Projects AS P
    LEFT JOIN Employee_Projects AS EP ON P.project_id = EP.project_id
    WHERE EP.employee_id IS NULL;
    """
)

# 26. List employees who have a salary greater than the average salary of their own department
execute_and_print_query(
    "26. Employees earning above their department's average",
    """
    SELECT
        E.first_name, E.last_name, D.department_name, S.amount AS employee_salary
    FROM Employees AS E
    JOIN Departments AS D ON E.department_id = D.department_id
    JOIN Salaries AS S ON E.employee_id = S.employee_id
    WHERE S.effective_date = (SELECT MAX(effective_date) FROM Salaries WHERE employee_id = E.employee_id)
      AND S.amount > (
            SELECT AVG(S2.amount)
            FROM Salaries AS S2
            JOIN Employees AS E2 ON S2.employee_id = E2.employee_id
            WHERE E2.department_id = E.department_id
              AND S2.effective_date = (SELECT MAX(effective_date) FROM Salaries WHERE employee_id = E2.employee_id)
        )
    ORDER BY D.department_name, employee_salary DESC;
    """
)

# 27. Count of 'Present', 'Absent', 'Leave' statuses per department for the last month
execute_and_print_query(
    "27. Monthly Attendance Summary per Department",
    """
    SELECT
        D.department_name,
        SUM(CASE WHEN A.status = 'Present' THEN 1 ELSE 0 END) AS present_count,
        SUM(CASE WHEN A.status = 'Absent' THEN 1 ELSE 0 END) AS absent_count,
        SUM(CASE WHEN A.status = 'Leave' THEN 1 ELSE 0 END) AS leave_count
    FROM Departments AS D
    JOIN Employees AS E ON D.department_id = E.department_id
    JOIN Attendance AS A ON E.employee_id = A.employee_id
    WHERE A.attendance_date BETWEEN date('now', '-1 month') AND date('now')
    GROUP BY D.department_name
    ORDER BY D.department_name;
    """
)

# 28. Employees who manage others but are not assigned to any projects themselves
execute_and_print_query(
    "28. Managers not on Projects",
    """
    SELECT
        M.first_name, M.last_name
    FROM Employees AS M
    WHERE M.employee_id IN (SELECT DISTINCT manager_id FROM Employees WHERE manager_id IS NOT NULL)
      AND M.employee_id NOT IN (SELECT DISTINCT employee_id FROM Employee_Projects);
    """
)

# 29. Projects completed within their budget (status is 'Completed' and actual cost <= budget)
# Note: We don't have 'actual cost' in the schema, so we'll simulate by checking if status is 'Completed'.
execute_and_print_query(
    "29. Completed Projects (assuming within budget if status 'Completed')",
    """
    SELECT
        project_name, budget, status
    FROM Projects
    WHERE status = 'Completed';
    """
)

# 30. Top 3 departments by average hours worked by their employees (last 30 days)
execute_and_print_query(
    "30. Top 3 Departments by Avg Hours Worked (last 30 days)",
    """
    SELECT
        D.department_name,
        AVG(A.hours_worked) AS average_hours_worked
    FROM Departments AS D
    JOIN Employees AS E ON D.department_id = E.department_id
    JOIN Attendance AS A ON E.employee_id = A.employee_id
    WHERE A.attendance_date BETWEEN date('now', '-30 days') AND date('now')
      AND A.hours_worked IS NOT NULL
    GROUP BY D.department_name
    ORDER BY average_hours_worked DESC
    LIMIT 3;
    """
)

# 31. Employees whose latest salary effective date is more than 1 year old (might need review)
execute_and_print_query(
    "31. Employees with outdated salary records",
    """
    SELECT
        E.first_name, E.last_name, S.amount, S.effective_date
    FROM Employees AS E
    JOIN Salaries AS S ON E.employee_id = S.employee_id
    WHERE S.effective_date = (SELECT MAX(effective_date) FROM Salaries WHERE employee_id = E.employee_id)
      AND S.effective_date < date('now', '-1 year')
    ORDER BY S.effective_date ASC
    LIMIT 5;
    """
)

# 32. Projects with a higher budget than the average budget of all projects
execute_and_print_query(
    "32. Projects with Above-Average Budgets",
    """
    SELECT
        project_name, budget
    FROM Projects
    WHERE budget > (SELECT AVG(budget) FROM Projects)
    ORDER BY budget DESC
    LIMIT 5;
    """
)

# 33. List employees who have never taken a 'Leave' (based on attendance records)
execute_and_print_query(
    "33. Employees who have never taken a leave",
    """
    SELECT
        E.first_name, E.last_name
    FROM Employees AS E
    LEFT JOIN Attendance AS A ON E.employee_id = A.employee_id AND A.status = 'Leave'
    GROUP BY E.employee_id
    HAVING COUNT(A.attendance_id) = 0;
    """
)

# 34. Find the department with the highest number of projects assigned to its employees
execute_and_print_query(
    "34. Department with Most Project Involvement",
    """
    SELECT
        D.department_name, COUNT(DISTINCT EP.project_id) AS num_distinct_projects
    FROM Departments AS D
    JOIN Employees AS E ON D.department_id = E.department_id
    JOIN Employee_Projects AS EP ON E.employee_id = EP.employee_id
    GROUP BY D.department_name
    ORDER BY num_distinct_projects DESC
    LIMIT 1;
    """
)



--- 1. Employees with salary outside job range ---
No results found.

--- 2. Department with the highest average salary ---
department_name | avg_dept_salary
-----------------------------------
Marketing | 140291.96

--- 3. Employees with no attendance in last 30 days ---
first_name | last_name
------------------------
Employee4 | Lastname4
Employee9 | Lastname9
Employee10 | Lastname10
Employee11 | Lastname11
Employee13 | Lastname13
Employee14 | Lastname14
Employee19 | Lastname19
Employee20 | Lastname20
Employee21 | Lastname21
Employee22 | Lastname22
Employee23 | Lastname23
Employee25 | Lastname25
Employee26 | Lastname26
Employee27 | Lastname27
Employee28 | Lastname28
Employee29 | Lastname29
Employee30 | Lastname30
Employee31 | Lastname31
Employee34 | Lastname34
Employee35 | Lastname35
Employee38 | Lastname38
Employee39 | Lastname39
Employee41 | Lastname41
Employee42 | Lastname42
Employee44 | Lastname44
Employee46 | Lastname46
Employee50 | Lastname50

--- 4. Overdue Active Projects --